# 03 — Complex<T> and DFT

## Usage and contracts
Charts migrated to PlotSharp through MathemaSharp.PlotSharp.csx: Release builds of MathemaSharp and PlotSharp are required. Runner: dotnet run --file ops/RunMigratedCoreNotebook.cs. Two SVGs validated as XML; layout and tooltips are not equivalent to the previous renderer, visual verification is still open.
Dft is forward and unnormalised, cost O(N^2), not an FFT; empty input returns an empty array. HermitianDot conjugates the first argument and is antilinear on the left; it requires equal lengths. VectorDot requires equal lengths, has a scalar fallback if Vector<T> is not supported and can differ from the scalar result because of the reduction order. Constraints: IEEE for VectorDot, IEEE + MinMax for Complex<T>.
[Full reference](12-api-reference.ipynb), [checks on the six formats](11-consolidation.ipynb).
**JupyterSharp** kernel (C#) on the **.NET 11** runtime. Prerequisite: `dotnet build src/MathemaSharp.slnx -c Release`. From a terminal: `scripts/run-notebooks.ps1` (sets the roll-forward to runtime 11); in VS Code you need `DOTNET_ROLL_FORWARD=LatestMajor` and, as long as .NET 11 is RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

`Complex<T>` (.NET 11) is generic over `T : IFloatingPointIeee754<T>, IMinMaxValue<T>`: the library uses it for `Signals.Dft<T>` and for the whole quantum part. There is no natural ordering. Angles are in radians. Forward DFT, unnormalised, O(N²), not an FFT.

In [ ]:
#load "../shared/MathemaSharp.PlotSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
var z = new Complex<double>(3,4);
Console.WriteLine($"{z}: modulo={z.GetMagnitude()}, fase={z.GetPhase()}");
Console.WriteLine(Complex<double>.Sqrt(-Complex<double>.One));
Console.WriteLine(Signals.HermitianDot<double>(new[] {z},new[] {z}));
Console.WriteLine(NumericAlgorithms.Dot<Complex<double>>(new[] {z},new[] {z}));

<3; 4>: modulo=5, fase=0.9272952180016122
<0; -1>
<25; 0>
<-7; 24>

In [ ]:
int n = 16;
var samples = Enumerable.Range(0,n).Select(i => new Complex<double>(Math.Cos(2*Math.PI*3*i/n),0)).ToArray();
var spectrum = Signals.Dft<double>(samples);
SpectrumChart.Show("|X[k]| — cosine at frequency 3 over 16 samples", spectrum.Select(c => c.GetMagnitude()).ToArray());
var leaking = Signals.Dft<double>(Enumerable.Range(0,n).Select(i => new Complex<double>(Math.Cos(2*Math.PI*3.5*i/n),0)).ToArray());
SpectrumChart.Show("|X[k]| — frequency 3.5: spectral leakage", leaking.Select(c => c.GetMagnitude()).ToArray());

|X[k]| — cosine at frequency 3 over 16 samples 0 2 4 6 8 0 5 10 15 k |X[k]| — cosine at frequency 3 over 16 samples · x=0: 1.2e-15 |X[k]| — cosine at frequency 3 over 16 samples · x=1: 1.0e-15 |X[k]| — cosine at frequency 3 over 16 samples · x=2: 2.2e-15 |X[k]| — cosine at frequency 3 over 16 samples · x=3: 8 |X[k]| — cosine at frequency 3 over 16 samples · x=4: 8.3e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=5: 6.3e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=6: 7.4e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=7: 6.7e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=8: 8.3e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=9: 7.2e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=10: 7.4e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=11: 9.4e-16 |X[k]| — cosine at frequency 3 over 16 samples · x=12: 1.2e-15 |X[k]| — cosine at frequency 3 over 16 samples · x=13: 8 |X[k]| — cosine at frequency 3 over 16 samples · x=14: 2.1e-15 |X[k]| — cosine at frequency 3 over 16 samples · x=15: 9.5e-16

|X[k]| — frequency 3.5: spectral leakage 0 2 4 6 0 5 10 15 k |X[k]| — frequency 3.5: spectral leakage · x=0: 1 |X[k]| — frequency 3.5: spectral leakage · x=1: 1.129 |X[k]| — frequency 3.5: spectral leakage · x=2: 1.705 |X[k]| — frequency 3.5: spectral leakage · x=3: 5.025 |X[k]| — frequency 3.5: spectral leakage · x=4: 5.222 |X[k]| — frequency 3.5: spectral leakage · x=5: 1.886 |X[k]| — frequency 3.5: spectral leakage · x=6: 1.271 |X[k]| — frequency 3.5: spectral leakage · x=7: 1.057 |X[k]| — frequency 3.5: spectral leakage · x=8: 1 |X[k]| — frequency 3.5: spectral leakage · x=9: 1.057 |X[k]| — frequency 3.5: spectral leakage · x=10: 1.271 |X[k]| — frequency 3.5: spectral leakage · x=11: 1.886 |X[k]| — frequency 3.5: spectral leakage · x=12: 5.222 |X[k]| — frequency 3.5: spectral leakage · x=13: 5.025 |X[k]| — frequency 3.5: spectral leakage · x=14: 1.705 |X[k]| — frequency 3.5: spectral leakage · x=15: 1.129

## Expected result
Peaks at k=3 and k=13, amplitude 8. With frequency 3.5 the energy spreads over all the bins: spectral leakage. Hermitian product with conjugation on the left: ⟨z,z⟩=25, bilinear product z·z=−7+24i. The same `Signals.Dft<T>` works with float and with Decimal32/64/128: see notebook 05 and the console's `--topic formats`.